<a href="https://colab.research.google.com/github/sarthak-geek/health_cost_with_linear_regression/blob/main/fcc_predict_health_costs_with_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
#converting categorical to numerical
dataset = pd.read_csv('insurance.csv')
for col in dataset.columns:
  if dataset[col].dtype == object:
    dataset[col] = layers.StringLookup(vocabulary = dataset[col].unique())(dataset[col])
#separating labels from dataset
labels = dataset.pop('expenses')
#spliting dataset and labels into train and test
train_ds, test_ds, train_labels, test_labels = train_test_split(dataset, labels, train_size = 0.8, random_state = 42)

In [ ]:
#creating scaler object for feature and label
feature_scaler = StandardScaler()
label_scaler = StandardScaler()
#always fit the scaler with training data, to prevent test data leakage
train_ds = feature_scaler.fit_transform(train_ds.values)
train_label = label_scaler.fit_transform(train_labels.values.reshape(-1,1))
test_dataset = feature_scaler.transform(test_ds.values)
test_labels = label_scaler.transform(test_labels.values.reshape(-1,1))

In [ ]:
model = keras.Sequential([layers.Dense(1)])
optimizer = keras.optimizers.Adam(learning_rate = 0.001)
model.compile( optimizer = optimizer, loss = 'mae', metrics = ['mae', 'mse'])

In [ ]:
history = model.fit(train_ds, train_label, epochs = 100, batch_size = 32)

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = label_scaler.inverse_transform(model.predict(test_dataset)).flatten() #since we had scaled labels, so we have to rescale the prediction to get the actual answer
test_labels = label_scaler.inverse_transform(test_labels)

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
